# Stage 5 — Analysis

Compute skill frequencies, placement lift, Wilson CIs, stratified rates, and temporal trends.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import ast
sys.path.insert(0, str(Path(".").resolve()))

from src.utils import (
    get_skill_frequency, get_top_skills, compute_placement_lift,
    wilson_confidence_interval, stratified_placement_analysis,
    select_temporal_granularity, skill_trend_over_time,
    correlation_matrix, detect_outliers_iqr,
)

In [ ]:
df_jobs = pd.read_csv("data/interim/jobs_canonicalized.csv", parse_dates=["date_posted"])
df_placements = pd.read_csv("data/interim/placements_canonicalized.csv")
df_jobs["canonical_skills"] = df_jobs["canonical_skills"].apply(ast.literal_eval)
df_placements["canonical_skills"] = df_placements["canonical_skills"].apply(ast.literal_eval)
skill_ds = pd.read_csv("data/processed/skill_demand_supply.csv")

## Top-20 skills by frequency

In [ ]:
freq = get_skill_frequency(df_jobs, "canonical_skills")
top20 = get_top_skills(freq, n=20)
print(top20)

## Placement lift with Wilson CI

In [ ]:
lift_df = compute_placement_lift(skill_ds, min_support=2)
print(f"Base placement rate: {lift_df["base_rate"].iloc[0]:.1%}")
lift_df

## Stratified analysis by experience

In [ ]:
strat = stratified_placement_analysis(df_placements, strat_col="years_of_experience")
print(strat)

## Temporal trend analysis

In [ ]:
granularity = select_temporal_granularity(df_jobs["date_posted"])
print(f"Selected granularity: {granularity}")
trend = skill_trend_over_time(df_jobs, "date_posted", resample_rule=granularity)
trend.head()

## Correlation matrix

In [ ]:
numeric_cols = [c for c in ["years_of_experience","offered_salary_lpa","placed"] if c in df_placements.columns]
corr = correlation_matrix(df_placements, numeric_cols)
print(corr)

## Salary outlier detection

In [ ]:
outliers = detect_outliers_iqr(df_jobs["salary_lpa"].dropna())
print(f"Outliers: {outliers.sum()} / {len(outliers)} ({outliers.mean():.1%})")

## Save analysis outputs

In [ ]:
lift_df.to_csv("data/output/placement_lift.csv", index=False)
strat.to_csv("data/output/stratified_rates.csv", index=False)
trend.to_csv("data/output/skill_trends.csv")
print("Saved.")